# INSERT Algorithm Evaluation

Developer notebook for analyzing an INSERT optimization run.
Points to a run directory and produces:
- **KPI comparison** — before vs after, with delta
- **Gantt 0** — raw API input (inferred timings, only on local runs)
- **Gantt 1** — preassigned schedule before the solver
- **Gantt 2** — inserted labors only, on an isolated canvas
- **Gantt 3** — full schedule with origin-coded colors
- **Detail table** — inserted labors with feasibility info

| Color | Meaning |
|-------|---------|
| 🔵 Blue | Labor — preassigned |
| 🟢 Green | Labor — inserted by INSERT |
| 🟠 Orange | Driver move (preassigned) |
| 🫒 Olive | Driver move (inserted) |
| ⬜ Grey | Free time |

In [1]:
from __future__ import annotations
import json
from collections import defaultdict
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

from alfred.analysis.solution_evaluation import (
    load_payload,
    flatten_labors,
    build_preassigned_lookup,
    reconstruct_timeline,
    compute_payload_summary,
    build_gantt_figure,
    build_service_distance_figure,
    build_driver_distance_figure,
    _parse_dt,
    BOGOTA_TZ,
)
from alfred.optimization.settings.model_params import ModelParams
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

_PROJECT_ROOT = Path("../..").resolve()
_params       = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh
GRACE_MINUTES:    int   = _params.tiempo_gracia_min
print(f"speed={ALFRED_SPEED_KMH} km/h | grace={GRACE_MINUTES} min")

speed=20.0 km/h | grace=15 min


In [2]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def csv_rows_to_analysis_rows(df: pd.DataFrame) -> list:
    """Convert preassigned_df.csv or output.csv rows to flat dicts for
    reconstruct_timeline / compute_payload_summary.

    Handles both CSV schemas:
    - output.csv       : labor_distance_km, driver_move_distance_km
    - preassigned_df   : dist_km (labor distance), computed_travel_min (move duration)
    """
    rows = []
    for _, r in df.iterrows():
        actual_start = _parse_dt(r.get("actual_start"))
        actual_end   = _parse_dt(r.get("actual_end"))
        duration_min = 0.0
        if actual_start and actual_end:
            duration_min = (actual_end - actual_start).total_seconds() / 60.0
        driver_id = r.get("assigned_driver")

        # Labor distance: output.csv uses labor_distance_km; preassigned_df uses dist_km
        labor_dist = float(r.get("labor_distance_km") or r.get("dist_km") or 0.0)

        # Move distance: output.csv stores it directly; preassigned_df stores computed_travel_min
        # reconstruct_timeline derives move_min = move_dist / speed, so invert: dist = min/60 * speed
        if pd.notna(r.get("driver_move_distance_km")) and float(r.get("driver_move_distance_km") or 0) > 0:
            move_dist = float(r["driver_move_distance_km"])
        elif pd.notna(r.get("computed_travel_min")) and float(r.get("computed_travel_min") or 0) > 0:
            move_dist = float(r["computed_travel_min"]) / 60.0 * ALFRED_SPEED_KMH
        else:
            move_dist = 0.0

        rows.append({
            "labor_id":                str(r.get("labor_id")),
            "service_id":              r.get("service_id"),
            "labor_type":              r.get("labor_type"),
            "labor_schedule_date":     r.get("schedule_date"),
            "service_labor_index":     r.get("labor_sequence", 0) or 0,
            "driver_id":               str(driver_id) if pd.notna(driver_id) else None,
            "actual_start":            actual_start,
            "actual_end":              actual_end,
            "duration_min":            duration_min,
            "labor_distance_km":       labor_dist,
            "driver_move_distance_km": move_dist,
            "is_infeasible":           bool(r.get("is_infeasible", False)),
            "reassignment_candidate":  bool(r.get("reassignment_candidate", False)),
            "original_assigned_driver": r.get("assigned_driver"),
            "map_start_wkt":           r.get("map_start_point") or None,
            "map_end_wkt":             r.get("map_end_point")   or None,
        })
    return rows
# ── Multi-color INSERT Gantt ──────────────────────────────────────────────────

_INSERT_COLORS = {
    "FREE_TIME":         "rgba(190, 190, 190, 0.35)",
    "preassigned_move":  "rgba(250, 155,  45, 0.85)",
    "inserted_move":     "rgba(120, 180,  80, 0.75)",
    "preassigned_labor": "rgba( 55, 115, 200, 0.88)",
    "inserted_labor":    "rgba( 40, 170,  80, 0.90)",
}
_INSERT_LABELS = {
    "FREE_TIME":         "Free time",
    "preassigned_move":  "Driver move (preassigned)",
    "inserted_move":     "Driver move (inserted)",
    "preassigned_labor": "Labor — preassigned",
    "inserted_labor":    "Labor — inserted",
}
_INSERT_ORDER = [
    "FREE_TIME", "preassigned_move", "inserted_move",
    "preassigned_labor", "inserted_labor",
]
_MIN_VT_MS = 5 * 60 * 1000


def _to_plotly_dt(ts) -> str:
    if ts is None:
        return ""
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        ts = ts.tz_convert("America/Bogota").tz_localize(None)
    return ts.isoformat()


def _classify_insert(seg_type: str, origin: str) -> str:
    if seg_type == "FREE_TIME":
        return "FREE_TIME"
    if seg_type == "DRIVER_MOVE":
        return "inserted_move" if origin == "inserted" else "preassigned_move"
    return "inserted_labor" if origin == "inserted" else "preassigned_labor"


def build_insert_gantt(segments_with_origin: list, driver_ids: list,
                       label: str = "INSERT") -> go.Figure:
    """Multi-color Gantt for INSERT. Segments must have an 'origin' field."""
    by_type = defaultdict(list)
    drv_set = set(driver_ids)
    for seg in segments_with_origin:
        if seg["driver_id"] in drv_set:
            key = _classify_insert(seg["segment_type"], seg.get("origin", "preassigned"))
            by_type[key].append(seg)

    fig = go.Figure()
    for seg_type in _INSERT_ORDER:
        segs = by_type.get(seg_type, [])
        if not segs:
            continue
        base_vals, x_vals, y_vals, hovers = [], [], [], []
        for s in segs:
            dur_ms = (s["end"] - s["start"]).total_seconds() * 1000
            if dur_ms < 0:
                continue
            is_labor = seg_type in ("preassigned_labor", "inserted_labor")
            if is_labor and dur_ms == 0:
                dur_ms = _MIN_VT_MS
            elif not is_labor and dur_ms == 0:
                continue
            base_vals.append(_to_plotly_dt(s["start"]))
            x_vals.append(dur_ms)
            y_vals.append(s["driver_id"])
            ht = (
                f"<b>{_INSERT_LABELS[seg_type]}</b><br>"
                f"Driver: {s['driver_id']}<br>"
                f"Labor: {s['labor_id']}<br>"
                f"Service: {s['service_id']}<br>"
                f"Start: {pd.Timestamp(s['start']).strftime('%H:%M')}<br>"
                f"End: {pd.Timestamp(s['end']).strftime('%H:%M')}<br>"
                f"Duration: {s['duration_min']:.1f} min"
            )
            if s.get("distance_km", 0) > 0:
                ht += f"<br>Distance: {s['distance_km']:.2f} km"
            if s.get("is_infeasible"):
                ht += "<br><i>⚠ infeasible</i>"
            hovers.append(ht)
        if not x_vals:
            continue
        fig.add_trace(go.Bar(
            x=x_vals, y=y_vals, base=base_vals, orientation="h",
            name=_INSERT_LABELS[seg_type],
            marker_color=_INSERT_COLORS[seg_type],
            hovertext=hovers, hoverinfo="text",
            showlegend=True, legendgroup=seg_type,
        ))

    all_segs_in_plot = [s for segs in by_type.values() for s in segs if s["driver_id"] in drv_set]
    pad = timedelta(minutes=30)
    if all_segs_in_plot:
        t_min = min(s["start"] for s in all_segs_in_plot)
        t_max = max(s["end"]   for s in all_segs_in_plot)
        fig.update_xaxes(type="date", range=[_to_plotly_dt(t_min - pad), _to_plotly_dt(t_max + pad)])
    fig.update_xaxes(tickformat="%H:%M", title_text="Time (Bogotá)")
    row_h = max(35, 600 // max(len(driver_ids), 1))
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=list(reversed(driver_ids)),
        title_text="Driver",
    )
    fig.update_layout(
        title_text=label, barmode="overlay",
        height=max(420, row_h * len(driver_ids) + 160),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="center", x=0.5),
        hovermode="closest",
        margin=dict(l=120, r=20, t=100, b=60),
    )
    return fig

In [3]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set RUN_DIR to a specific INSERT run directory, or leave None to auto-pick.
RUN_DIR: Optional[Path] = None
ALGORITHM_FILTER: str   = "INSERT"   # used for auto-pick filtering

_runs_base = _PROJECT_ROOT / "data" / "runs"
if RUN_DIR is None:
    _candidates = sorted(
        [
            d for d in _runs_base.iterdir()
            if d.is_dir()
            and (d / "run.json").exists()
            and (d / "output" / "output.csv").exists()
        ],
        key=lambda d: d.stat().st_mtime,
        reverse=True,
    )
    _matching = []
    for _d in _candidates:
        try:
            _mf = json.loads((_d / "run.json").read_text())
            if str(_mf.get("solver", "")).upper() == ALGORITHM_FILTER:
                _matching.append(_d)
        except Exception:
            pass
    if not _matching:
        print(f"[warning] No completed runs found with solver={ALGORITHM_FILTER!r}.")
        print("Available runs:")
        for _d in _candidates[:5]:
            _solver = ""
            try:
                _solver = json.loads((_d / "run.json").read_text()).get("solver", "?")
            except Exception:
                pass
            print(f"  {_d.name}  solver={_solver}")
        print("\nSet RUN_DIR manually to proceed.")
        RUN_DIR = _candidates[0] if _candidates else None
    else:
        RUN_DIR = _matching[0]

if RUN_DIR is None:
    raise FileNotFoundError("No suitable run found. Set RUN_DIR manually.")

LABEL_BEFORE = "before"
LABEL_AFTER  = "after (insert)"
print(f"Run dir : {RUN_DIR}")

Run dir : /Users/jbeta/Documents/AlfredProject/AlfredDEV/data/runs/run-ass-a3258d7b


In [4]:
# ── Run manifest ──────────────────────────────────────────────────────────────
_manifest = json.loads((RUN_DIR / "run.json").read_text())
print(json.dumps(_manifest, indent=2, default=str))

{
  "run_id": "ass-a3258d7b",
  "started_at": "2026-04-19T15:25:42.989359+02:00",
  "status": "success",
  "finished_at": "2026-04-19T15:27:43.963789+02:00",
  "duration_seconds": 121.0,
  "solver": "INSERT",
  "services": {
    "total": 40,
    "planned": 40,
    "failed": 0
  },
  "labors": {
    "total": 44,
    "preassigned": 40,
    "processed": 4,
    "assigned": 44
  },
  "instance": {
    "department": "25",
    "start_date": "2026-04-17",
    "end_date": "2026-04-17"
  },
  "algorithm": {
    "name": "INSERT",
    "max_iterations": {
      "25": 750
    },
    "n_processes": null,
    "parallel": false,
    "time_previous_freeze": null
  },
  "stage_timings_seconds": {
    "load_request": 0.045,
    "input_acquisition": 1.098,
    "parse_input": 0.033,
    "validate_input": 0.005,
    "driver_directory_load": 0.224,
    "reconstruct_preassigned": 4.234,
    "solve": 114.87,
    "validate_solution": 0.023,
    "evaluate_solution": 0.029,
    "format_output": 0.021,
    "save_wa

In [5]:
# ── Load CSV files ────────────────────────────────────────────────────────────
_pre_csv  = RUN_DIR / "intermediate" / "preassigned_df.csv"
_out_csv  = RUN_DIR / "output" / "output.csv"

_pre_df = pd.read_csv(_pre_csv, low_memory=False) if _pre_csv.exists() else pd.DataFrame()
_out_df = pd.read_csv(_out_csv, low_memory=False)

for _df in (_pre_df, _out_df):
    for _col in ("actual_start", "actual_end", "schedule_date"):
        if _col in _df.columns:
            _df[_col] = _df[_col].apply(lambda x: pd.Timestamp(x) if pd.notna(x) else pd.NaT)

if _pre_df.empty or "labor_id" not in _pre_df.columns:
    print("[warning] preassigned_df is empty — 'Before' sections will be skipped.")

print(f"preassigned_df : {len(_pre_df)} rows")
print(f"output.csv     : {len(_out_df)} rows")

preassigned_df : 40 rows
output.csv     : 44 rows


In [6]:
# ── Build row dicts and classify origins ──────────────────────────────────────
_preassigned_labor_ids = set(
    _pre_df["labor_id"].dropna().astype(str)
) if not _pre_df.empty and "labor_id" in _pre_df.columns else set()

rows_before = csv_rows_to_analysis_rows(_pre_df) if not _pre_df.empty else []
rows_after  = csv_rows_to_analysis_rows(_out_df)

for r in rows_before:
    r["origin"] = "preassigned"
for r in rows_after:
    r["origin"] = "preassigned" if r["labor_id"] in _preassigned_labor_ids else "inserted"

_inserted_labor_ids = {r["labor_id"] for r in rows_after if r["origin"] == "inserted"}

print(f"rows_before : {len(rows_before)}")
print(f"rows_after  : {len(rows_after)}")
print(f"  preassigned : {sum(1 for r in rows_after if r['origin'] == 'preassigned')}")
print(f"  inserted    : {len(_inserted_labor_ids)}")

rows_before : 40
rows_after  : 44
  preassigned : 40
  inserted    : 4


In [7]:
# ── Reconstruct timelines ─────────────────────────────────────────────────────
segments_before = reconstruct_timeline(rows_before, ALFRED_SPEED_KMH) if rows_before else []
segments_after  = reconstruct_timeline(rows_after,  ALFRED_SPEED_KMH)

_origin_map = {r["labor_id"]: r["origin"] for r in rows_after}
for r in rows_before:
    _origin_map[r["labor_id"]] = "preassigned"

for seg in segments_before:
    seg["origin"] = "preassigned"
for seg in segments_after:
    seg["origin"] = _origin_map.get(seg["labor_id"], "preassigned")

drivers_before = sorted({r["driver_id"] for r in rows_before if r["driver_id"]})
drivers_after  = sorted({r["driver_id"] for r in rows_after  if r["driver_id"]})
all_services   = sorted({str(r["service_id"]) for r in rows_before + rows_after if r["service_id"] is not None})

print(f"segments_before : {len(segments_before)} | drivers: {len(drivers_before)}")
print(f"segments_after  : {len(segments_after)} | drivers: {len(drivers_after)}")

segments_before : 95 | drivers: 13
segments_after  : 101 | drivers: 13


---
## KPI Comparison — Before vs After

In [8]:
# ── KPI comparison ────────────────────────────────────────────────────────────
_kpi_before = compute_payload_summary(
    rows_before, tiempo_gracia_min=GRACE_MINUTES, segments=segments_before
) if rows_before else {}
_kpi_after = compute_payload_summary(
    rows_after, tiempo_gracia_min=GRACE_MINUTES, segments=segments_after
)

def _delta(a, b):
    try:
        return round(float(b) - float(a), 4)
    except (TypeError, ValueError):
        return None

def _delta_pct(a, b):
    try:
        fa, fb = float(a), float(b)
        return round(100.0 * (fb - fa) / fa, 2) if fa != 0 else None
    except (TypeError, ValueError):
        return None

_METRICS = [
    ("services_count",                "services"),
    ("labors_count",                  "labors_total"),
    ("labors_vt_count",               "labors_vt"),
    ("drivers_count",                 "drivers_used"),
    ("labors_assigned",               "labors_assigned"),
    ("labors_preassigned",            "labors_preassigned"),
    ("labors_infeasible",             "labors_infeasible"),
    ("labors_infeasible_pct",         "labors_infeasible_pct"),
    ("labors_in_grace",               "labors_in_grace"),
    ("total_grace_min",               "total_grace_min"),
    ("total_labor_distance_km",       "total_labor_distance_km"),
    ("total_driver_move_distance_km", "total_driver_move_distance_km"),
    ("total_distance_km",             "total_distance_km"),
    ("avg_labor_distance_km",         "avg_labor_distance_km"),
    ("avg_driver_move_distance_km",   "avg_driver_move_distance_km"),
]

pd.DataFrame([
    {
        "metric":      label,
        LABEL_BEFORE:  _kpi_before.get(key),
        LABEL_AFTER:   _kpi_after.get(key),
        "delta":       _delta(_kpi_before.get(key), _kpi_after.get(key)),
        "delta_pct":   _delta_pct(_kpi_before.get(key), _kpi_after.get(key)),
    }
    for key, label in _METRICS
]).set_index("metric")

,before,after (insert),delta,delta_pct
metric,,,,
services,34.00,36.00,2.00,5.88
labors_total,40.00,44.00,4.00,10.00
labors_vt,37.00,40.00,3.00,8.11
drivers_used,13.00,13.00,0.00,0.00
labors_assigned,37.00,40.00,3.00,8.11
labors_preassigned,40.00,44.00,4.00,10.00
labors_infeasible,0.00,2.00,2.00,NaN
labors_infeasible_pct,0.00,4.55,4.55,NaN
labors_in_grace,3.00,2.00,-1.00,-33.33


---
## Gantt 1 — Before (Preassigned Schedule)

In [9]:
if rows_before:
    build_gantt_figure(
        segments_before, drivers_before,
        f"Before — preassigned schedule — {RUN_DIR.name}",
    ).show()
else:
    print("[info] No preassigned schedule to display (preassigned_df is empty).")

---
## Gantt 2 — Inserted Labors (Isolated)
_Each inserted labor shown on its own virtual driver row — no move context._  
_Useful for understanding what the solver placed and when, independent of the existing schedule._

In [10]:
_inserted_rows = [r for r in rows_after if r["origin"] == "inserted"]

if _inserted_rows:
    _virtual_rows = []
    for r in sorted(_inserted_rows, key=lambda x: (x["actual_start"] or datetime.min)):
        _vrow = dict(r)
        _vrow["driver_id"]               = f"NEW:{r['labor_id']}"
        _vrow["driver_move_distance_km"]  = 0.0
        _virtual_rows.append(_vrow)

    _virtual_segs = reconstruct_timeline(_virtual_rows, ALFRED_SPEED_KMH)
    _vt_segs      = [s for s in _virtual_segs if s["segment_type"] == "VEHICLE_TRANSPORTATION"]
    _virtual_drv  = [r["driver_id"] for r in _virtual_rows]  # preserve order

    build_gantt_figure(
        _vt_segs, _virtual_drv,
        f"Inserted Labors (isolated) — {RUN_DIR.name}",
    ).show()
else:
    print("[info] No inserted labors found.")

---
## Gantt 3 — Full Schedule (Color by Origin)

| Color | Meaning |
|-------|---------|
| 🔵 Blue | Labor — preassigned |
| 🟢 Green | Labor — inserted |
| 🟠 Orange | Driver move (preassigned) |
| 🫒 Olive | Driver move (inserted) |
| ⬜ Grey | Free time |

In [11]:
build_insert_gantt(
    segments_after, drivers_after,
    label=f"Full Schedule (INSERT) — {RUN_DIR.name}",
).show()

---
## Distance per Service / Driver

In [12]:
build_service_distance_figure(rows_after, all_services, LABEL_AFTER).show()
build_driver_distance_figure(rows_after, drivers_after, LABEL_AFTER).show()

---
## Inserted Labor Detail

In [13]:
_inserted_detail = _out_df[
    _out_df["labor_id"].astype(str).isin(_inserted_labor_ids)
].copy()

_detail_cols = [c for c in [
    "service_id", "labor_id", "labor_type", "assigned_driver",
    "schedule_date", "actual_start", "actual_end",
    "is_infeasible", "infeasibility_cause_code",
    "labor_distance_km", "driver_move_distance_km",
] if c in _inserted_detail.columns]

if _inserted_detail.empty:
    print("[info] No inserted labors to display.")
else:
    _inserted_detail[_detail_cols].sort_values(
        ["assigned_driver", "actual_start"]
    ).reset_index(drop=True)